In [28]:
import torch
x = torch.randn(5, requires_grad=True)
y = x.pow(2)
print(x.equal(y.grad_fn._saved_self))  # True
print(x is y.grad_fn._saved_self)  # True

True
True


In [ ]:
y.grad_fn._saved_self 
# this returns the intermediate tensor that torch saved from the forward pass, 
# which is used in the backward pass to compute gradients. 
# In this case, it returns the original tensor `x` that was used to compute `y`.


tensor([ 0.8100, -0.7058, -0.3022,  0.7219, -0.7589], requires_grad=True)

In [12]:
x

tensor([ 0.8100, -0.7058, -0.3022,  0.7219, -0.7589], requires_grad=True)

In [15]:
x = torch.randn(5, requires_grad=True)
y = x.exp()
print(y.equal(y.grad_fn._saved_result))  # True
print(y is y.grad_fn._saved_result)  # False

True
False


In [16]:
x = torch.tensor([1., 1.], requires_grad=True)
div = torch.tensor([0., 1.])

y = x / div          # Results in [inf, 1]
mask = div != 0      # [False, True]
loss = y[mask].sum()
loss.backward()
print(x.grad)        # [nan, 1], not [0, 1]

tensor([nan, 1.])


In [27]:
x = torch.tensor([1., 1.], requires_grad=True)
div = torch.tensor([0., 1.])

mask = div != 0             # [False, True]
safe = torch.zeros_like(x) #  [0,    0]
safe[mask] = x[mask] / div[mask]
loss = safe.sum() # Results in 1, not inf
loss.backward()      # Produces safe gradients [0, 1]

In [26]:
safe

tensor([0., 1.], grad_fn=<IndexPutBackward0>)

In [20]:
x[mask]

tensor([1.], grad_fn=<IndexBackward0>)

In [21]:
div[mask]

tensor([1.])

In [34]:
x = torch.tensor(1.0, requires_grad=True)
y1 = x.pow(2)
y1.backward()
print(x.grad)  # 2.0

tensor(2.)


In [35]:
y2 = x.pow(3)
y2.backward()
print(x.grad)  # 2.0 + 3.0 = 5.0

tensor(5.)


In [1]:
import torch
import torch.nn as nn

# Set k (hidden layer width) for simulation
k = 3

class SimpleReLU(nn.Module):
    def __init__(self, k_dim):
        super(SimpleReLU, self).__init__()
        # W1: Matrix in R^(k x 1) - weight matrix for the first layer
        # Here, it maps 1 input feature to k hidden units
        self.W1 = nn.Parameter(torch.randn(k_dim, 1)) # Shape: [k, 1]

        # b1: Vector in R^(k x 1) - bias vector for the first layer
        # Correctly initialized with dimension k, matching the number of hidden units
        self.b1 = nn.Parameter(torch.randn(k_dim, 1)) # Shape: [k, 1]

        # W2: Matrix in R^(1 x k) - weight matrix for the output layer
        # Maps k hidden features back to 1 output value
        self.W2 = nn.Parameter(torch.randn(1, k_dim)) # Shape: [1, k]

        # b2: Vector in R^(1 x 1) - bias scalar for the output layer
        self.b2 = nn.Parameter(torch.randn(1, 1))    # Shape: [1, 1]

        self.relu = nn.ReLU()

    def forward(self, x):
        # x is expected to be a single scalar input, but PyTorch tensors are general.
        # Let's ensure x has the correct [1, 1] or equivalent shape if passed as a scalar
        if x.dim() == 0:
             x = x.unsqueeze(0).unsqueeze(0) # Convert scalar 1 to shape [1, 1]
        elif x.dim() == 1 and x.shape[0] == 1:
             x = x.unsqueeze(0)             # Convert [1] to shape [1, 1]
        elif x.dim() > 2 or (x.dim()==2 and x.shape != (1,1)):
             # Optional: add proper shape error handling if needed,
             # but for simulation we assume 1x1 input
             pass

        print(f"Input x shape: {x.shape}")

        # W1x + b1:
        # (W1: k x 1) * (x: 1 x 1) -> product is shape [k, 1]
        # (product: k x 1) + (b1: k x 1) -> result is shape [k, 1]
        linear1_out = torch.mm(self.W1, x) + self.b1
        print(f"Pre-activation shape (W1x + b1): {linear1_out.shape}")

        # ReLU(W1x + b1):
        # Activation function applies element-wise, shape remains [k, 1]
        act1_out = self.relu(linear1_out)
        print(f"Activation shape: {act1_out.shape}")

        # W2 ReLU(...) + b2:
        # (W2: 1 x k) * (act: k x 1) -> product is shape [1, 1] (scalar)
        # (product: 1 x 1) + (b2: 1 x 1) -> final output is shape [1, 1]
        output = torch.mm(self.W2, act1_out) + self.b2
        print(f"Output f(x) shape: {output.shape}")

        return output

# --- Simulation ---
torch.manual_seed(42) # Set seed for reproducibility

# 1. Instantiate the model with k=3
model = SimpleReLU(k)
print("Model initialized with k=3")

# 2. Simulate with a single scalar input (e.g., 2.5)
scalar_input = torch.tensor(2.5)
print(f"Simulating for scalar input: {scalar_input.item()}")
model_output = model(scalar_input)
print(f"Model f(2.5) = {model_output.item():.4f}")

print("-" * 20)

# 3. Experiment with a larger k (e.g., 5)
k_large = 5
model_large = SimpleReLU(k_large)
print(f"Model initialized with k={k_large}")
print(f"Simulating for scalar input: {scalar_input.item()}")
model_output_large = model_large(scalar_input)
print(f"Model_large f(2.5) = {model_output_large.item():.4f}")

print("-" * 20)

# 4. Illustrate parameters dimensions are fixed after initialization
print(f"Model W1 shape (k=3): {model.W1.shape}")
print(f"Model b1 shape (k=3): {model.b1.shape}")
print(f"Model W2 shape (k=3): {model.W2.shape}")
print(f"Model b2 shape (k=3): {model.b2.shape}")
print(f"Model_large W1 shape (k=5): {model_large.W1.shape}")

Model initialized with k=3
Simulating for scalar input: 2.5
Input x shape: torch.Size([1, 1])
Pre-activation shape (W1x + b1): torch.Size([3, 1])
Activation shape: torch.Size([3, 1])
Output f(x) shape: torch.Size([1, 1])
Model f(2.5) = 2.8193
--------------------
Model initialized with k=5
Simulating for scalar input: 2.5
Input x shape: torch.Size([1, 1])
Pre-activation shape (W1x + b1): torch.Size([5, 1])
Activation shape: torch.Size([5, 1])
Output f(x) shape: torch.Size([1, 1])
Model_large f(2.5) = 4.2791
--------------------
Model W1 shape (k=3): torch.Size([3, 1])
Model b1 shape (k=3): torch.Size([3, 1])
Model W2 shape (k=3): torch.Size([1, 3])
Model b2 shape (k=3): torch.Size([1, 1])
Model_large W1 shape (k=5): torch.Size([5, 1])
